# Step 3C — Zonal Market Model (updated with distributed wind)

This notebook updates the zonal model to make the nodal vs. zonal comparison more meaningful.

Main changes:
- six wind farms are distributed across the network at buses 3, 5, 7, 16, 21, and 23
- wind availability is bus-specific instead of a single aggregated wind node
- three line capacities are tightened to create more realistic congestion
- nodal demand is first distributed to buses and then aggregated to zones
- ATC is computed from the actual interzonal lines

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Conventional generators and network

In [ ]:
# Conventional generators
G = list(range(12))

P_max = {
    0: 152, 1: 152, 2: 350, 3: 591, 4: 60, 5: 155,
    6: 155, 7: 400, 8: 400, 9: 300, 10: 310, 11: 350
}

C = {
    0: 13.32, 1: 13.32, 2: 20.70, 3: 20.93, 4: 26.11,
    5: 10.52, 6: 10.52, 7: 6.02, 8: 5.47, 9: 0.00,
    10: 10.52, 11: 10.89
}

# Single-hour demand and demand bid
D_total = 2650.5
U_bid = 100.0

# Buses
buses = list(range(1, 25))

# Generator-to-bus mapping
gen_bus = {
    0: 1,  1: 2,  2: 7,  3: 13, 4: 15, 5: 15,
    6: 16, 7: 18, 8: 21, 9: 22, 10: 23, 11: 23
}

# Line data with updated bottlenecks
line_data = [
    (1,2,175), (1,3,175), (1,5,350),
    (2,4,175), (2,6,175),
    (3,9,175), (3,24,400),
    (4,9,175),
    (5,10,350),
    (6,10,175),
    (7,8,350),
    (8,9,175), (8,10,175),
    (9,11,400), (9,12,400),
    (10,11,400), (10,12,400),
    (11,13,500), (11,14,500),
    (12,13,500), (12,23,500),
    (13,23,250),   # updated from 500 to 250
    (14,16,250),   # updated from 500 to 250
    (15,16,500), (15,21,400), (15,24,500),  # updated from 1000 to 400
    (16,17,500), (16,19,500),
    (17,18,500), (17,22,500),
    (18,21,1000),
    (19,20,1000),
    (20,23,1000),
    (21,22,500)
]

print("Number of generators:", len(G))
print("Number of buses:", len(buses))
print("Number of lines:", len(line_data))

## 2. Zone definition

We keep the same 2-zone split used earlier:
- Zone A: buses 1–12
- Zone B: buses 13–24

In [ ]:
ZONES = ["A", "B"]
bus_zone = {b: ("A" if b <= 12 else "B") for b in buses}

zone_buses = {
    z: [b for b in buses if bus_zone[b] == z]
    for z in ZONES
}
zone_buses

## 3. Distributed wind farms

Updated IEEE RTS-style setup:
- wind farms at buses 3, 5, 7, 16, 21, and 23
- each wind farm has 200 MW installed capacity

In [ ]:
wind_nodes = [3, 5, 7, 16, 21, 23]
wind_cap = {n: 200.0 for n in wind_nodes}

pd.DataFrame({
    "wind_bus": wind_nodes,
    "capacity_MW": [wind_cap[n] for n in wind_nodes],
    "zone": [bus_zone[n] for n in wind_nodes]
})

## 4. Bus-specific wind profiles

The profiles are deterministic and only meant to be plausible:
- higher wind in the morning
- lower wind around peak demand hours
- small differences between locations

In [ ]:
hours = list(range(24))

base_cf = np.array([
    0.55, 0.58, 0.60, 0.62, 0.65, 0.68,
    0.72, 0.70, 0.62, 0.50, 0.40, 0.32,
    0.28, 0.25, 0.22, 0.20, 0.18, 0.16,
    0.18, 0.22, 0.30, 0.38, 0.45, 0.50
])

wind_cf = {
    3:  np.clip(base_cf * 0.95, 0, 1),
    5:  np.clip(base_cf * 1.05, 0, 1),
    7:  np.clip(np.roll(base_cf,  1) * 0.90, 0, 1),
    16: np.clip(np.roll(base_cf, -1) * 1.00, 0, 1),
    21: np.clip(base_cf * 1.10, 0, 1),
    23: np.clip(np.roll(base_cf,  2) * 0.92, 0, 1),
}

wind_avail = {
    n: {t: wind_cap[n] * wind_cf[n][t] for t in hours}
    for n in wind_nodes
}

# Visualize the wind profiles
plt.figure(figsize=(9, 5))
for n in wind_nodes:
    plt.plot(hours, [wind_avail[n][t] for t in hours], marker="o", linewidth=1.8, label=f"Bus {n}")
plt.xlabel("Hour")
plt.ylabel("Available wind power (MW)")
plt.title("Distributed wind availability profiles")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

## 5. Choose the study hour

In [ ]:
# Change this if you want to study another hour
t_star = 17

print("Selected hour:", t_star)
print("Wind availability at selected hour:")
for n in wind_nodes:
    print(f"Bus {n}: {wind_avail[n][t_star]:.2f} MW")

## 6. Nodal demand distribution and zonal demand aggregation

We first distribute the total system demand to load buses, then aggregate it to zones.

In [ ]:
load_percent = {
    1:3.8, 2:3.4, 3:6.3, 4:2.6, 5:2.5,
    6:4.8, 7:4.4, 8:6.0, 9:6.1, 10:6.8,
    13:9.3, 14:6.8, 15:11.1, 16:3.5,
    18:11.7, 19:6.4, 20:4.5
}

load_share = {n: load_percent[n] / 100 for n in load_percent}
print("Sum of load shares:", sum(load_share.values()))

D_bus = {n: D_total * load_share.get(n, 0.0) for n in buses}

demand_in_zone = {z: 0.0 for z in ZONES}
for b in buses:
    demand_in_zone[bus_zone[b]] += D_bus[b]

bid_in_zone = {z: U_bid for z in ZONES}

print("Demand in zones:")
for z in ZONES:
    print(f"Zone {z}: {demand_in_zone[z]:.2f} MW")

## 7. Aggregate wind to zones for the selected hour

In [ ]:
wind_in_zone = {z: 0.0 for z in ZONES}
for n in wind_nodes:
    z = bus_zone[n]
    wind_in_zone[z] += wind_avail[n][t_star]

print("Wind in zones:")
for z in ZONES:
    print(f"Zone {z}: {wind_in_zone[z]:.2f} MW")

## 8. Compute ATC from interzonal lines

In [ ]:
def compute_atc_from_line_data(line_data, bus_zone):
    atc = {}
    crossing_lines = []

    for (i, j, cap) in line_data:
        zi = bus_zone[i]
        zj = bus_zone[j]

        if zi != zj:
            key = tuple(sorted((zi, zj)))
            atc[key] = atc.get(key, 0.0) + float(cap)
            crossing_lines.append((i, j, cap, zi, zj))

    return atc, crossing_lines

atc, crossing_lines = compute_atc_from_line_data(line_data, bus_zone)

print("ATC between zones:", atc)
print("\nInterzonal lines:")
for row in crossing_lines:
    print(row)

## 9. Zonal market-clearing model

In [ ]:
def solve_zonal_market(
    ZONES, atc, G, C, P_max, gen_bus, bus_zone,
    demand_in_zone, bid_in_zone, wind_in_zone,
    atc_scale=1.0, verbose=False
):
    m = gp.Model("ZonalMarket")
    m.Params.OutputFlag = 1 if verbose else 0

    # Conventional generation
    p = m.addVars(G, lb=0.0, ub={i: P_max[i] for i in G}, name="p")

    # Wind dispatched per zone
    pw = m.addVars(ZONES, lb=0.0, ub=wind_in_zone, name="pw")

    # Demand served per zone
    d = m.addVars(ZONES, lb=0.0, ub=demand_in_zone, name="d")

    # Interzonal flows
    flow = {}
    for (a, b), cap in atc.items():
        cap_eff = atc_scale * float(cap)
        flow[(a, b)] = m.addVar(lb=-cap_eff, ub=cap_eff, name=f"f_{a}_{b}")

    # Objective: minimize cost - utility
    m.setObjective(
        gp.quicksum(C[i] * p[i] for i in G)
        - gp.quicksum(bid_in_zone[z] * d[z] for z in ZONES),
        GRB.MINIMIZE
    )

    # Zonal balances
    balance = {}
    for z in ZONES:
        gen_in_z = gp.quicksum(p[i] for i in G if bus_zone[gen_bus[i]] == z)

        net_import = 0
        for (a, b), fvar in flow.items():
            # positive f_(a,b): power from a to b
            if z == a:
                net_import += -fvar
            elif z == b:
                net_import += fvar

        balance[z] = m.addConstr(
            gen_in_z + pw[z] + net_import == d[z],
            name=f"balance_{z}"
        )

    m.optimize()

    if m.status != GRB.OPTIMAL:
        raise RuntimeError(f"Model not optimal. Status = {m.status}")

    prices = {z: balance[z].Pi for z in ZONES}
    flow_val = {k: v.X for k, v in flow.items()}
    total_cost = sum(C[i] * p[i].X for i in G)
    total_utility = sum(bid_in_zone[z] * d[z].X for z in ZONES)
    welfare = total_utility - total_cost

    return {
        "prices": prices,
        "flow_val": flow_val,
        "total_cost": total_cost,
        "welfare": welfare,
        "gen": {i: p[i].X for i in G},
        "wind": {z: pw[z].X for z in ZONES},
        "demand": {z: d[z].X for z in ZONES},
    }

## 10. Run the zonal model for the selected hour

In [ ]:
res = solve_zonal_market(
    ZONES=ZONES,
    atc=atc,
    G=G,
    C=C,
    P_max=P_max,
    gen_bus=gen_bus,
    bus_zone=bus_zone,
    demand_in_zone=demand_in_zone,
    bid_in_zone=bid_in_zone,
    wind_in_zone=wind_in_zone,
    atc_scale=1.0,
    verbose=False
)

print("Selected hour:", t_star)
print("Zonal prices (EUR/MWh):", res["prices"])
print("Interzonal flows (MW):", res["flow_val"])
print(f"Total generation cost (EUR): {res['total_cost']:.2f}")
print(f"Social welfare (EUR): {res['welfare']:.2f}")
print("Wind dispatched by zone (MW):", res["wind"])

## 11. ATC sensitivity analysis

In [ ]:
scales = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 10.0]

rows = []
for s in scales:
    out = solve_zonal_market(
        ZONES=ZONES,
        atc=atc,
        G=G,
        C=C,
        P_max=P_max,
        gen_bus=gen_bus,
        bus_zone=bus_zone,
        demand_in_zone=demand_in_zone,
        bid_in_zone=bid_in_zone,
        wind_in_zone=wind_in_zone,
        atc_scale=s,
        verbose=False
    )

    row = {
        "ATC_scale": s,
        "Welfare": out["welfare"],
        "Cost": out["total_cost"]
    }

    for z in ZONES:
        row[f"lambda_{z}"] = out["prices"][z]

    for (a, b), v in out["flow_val"].items():
        row[f"flow_{a}_{b}"] = v

    rows.append(row)

df = pd.DataFrame(rows)
df

## 12. Plot zonal prices vs. ATC

In [ ]:
plt.figure(figsize=(7, 5))

for z in ZONES:
    plt.plot(
        df["ATC_scale"],
        df[f"lambda_{z}"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=f"Zone {z}"
    )

plt.xlabel("ATC scale (multiplier)")
plt.ylabel("Zonal price (EUR/MWh)")
plt.title("Zonal prices as a function of ATC")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 13. Optional: summary tables for the selected hour

In [ ]:
gen_table = pd.DataFrame({
    "generator": list(G),
    "bus": [gen_bus[i] for i in G],
    "zone": [bus_zone[gen_bus[i]] for i in G],
    "Pmax_MW": [P_max[i] for i in G],
    "cost_EUR_per_MWh": [C[i] for i in G],
    "dispatch_MW": [res["gen"][i] for i in G]
})
gen_table

In [ ]:
wind_table = pd.DataFrame({
    "wind_bus": wind_nodes,
    "zone": [bus_zone[n] for n in wind_nodes],
    "capacity_MW": [wind_cap[n] for n in wind_nodes],
    f"available_MW_hour_{t_star}": [wind_avail[n][t_star] for n in wind_nodes]
})
wind_table

## 14. Placeholder for nodal-vs-zonal comparison

Paste the nodal bus prices from the corrected nodal notebook when they are ready.

In [ ]:
def zonal_average_from_nodal(lambda_nodal, bus_zone):
    zones = sorted(set(bus_zone.values()))
    zavg = {}
    for z in zones:
        buses_in_z = [b for b, zz in bus_zone.items() if zz == z]
        zavg[z] = float(np.mean([lambda_nodal[b] for b in buses_in_z]))
    return zavg

# Example usage after you paste nodal prices:
# lambda_nodal = {1: ..., 2: ..., ..., 24: ...}
# zavg = zonal_average_from_nodal(lambda_nodal, bus_zone)
# print("Average nodal price per zone:", zavg)
# print("Zonal market prices:", res["prices"])